CELL 1 — Imports and Setup

In [ ]:
import subprocess, sys

packages = ['mlflow', 'optuna', 'xgboost',
            'lightgbm', 'torch', 'darts']

for pkg in packages:
    try:
        __import__(pkg)
        print(f"✅ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable,
            '-m', 'pip', 'install', pkg, '--quiet'])
        print(f"✅ {pkg} installed")

In [ ]:
# ================================
# PHASE 3: MODEL DEVELOPMENT & TRAINING
# IRS-1: AI-Driven Demand Intelligence
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import os
import pickle
import json
import mlflow
import mlflow.sklearn
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

import sys
sys.path.append(
    'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'
)
from config import PROCESSED_PATH, MODELS_PATH
from sklearn.metrics import (mean_absolute_error,
                              mean_squared_error)

os.makedirs(str(MODELS_PATH), exist_ok=True)
mlflow.set_experiment("IRS1-Demand-Forecasting")

print("✅ Libraries loaded")
print(f"Models path: {MODELS_PATH}")
print(f"MLflow tracking ready")

In [ ]:
# ── Load corrected Phase 2 dataset ───────────────────────────────
df = pd.read_csv(
    str(PROCESSED_PATH / 'demand_features_v2.csv')
)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store_id', 'product_id', 'date'])
df = df.reset_index(drop=True)

# Load feature columns
with open(str(PROCESSED_PATH / 'feature_columns.json')) as f:
    ML_FEATURES = json.load(f)

# Keep only features that exist in df
ML_FEATURES = [f for f in ML_FEATURES if f in df.columns]

print(f"✅ Dataset loaded   : {df.shape}")
print(f"✅ ML Features      : {len(ML_FEATURES)} columns")
print(f"Date range: {df['date'].min()} "
      f"→ {df['date'].max()}")

In [ ]:
# ── Temporal Split ────────────────────────────────────────────────
df_sorted = df.sort_values('date').reset_index(drop=True)
n         = len(df_sorted)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = df_sorted.iloc[:train_end].copy()
val   = df_sorted.iloc[train_end:val_end].copy()
test  = df_sorted.iloc[val_end:].copy()

print("=" * 50)
print("TEMPORAL TRAIN / VAL / TEST SPLIT")
print("=" * 50)
print(f"Train : {len(train):,} rows "
      f"({train['date'].min().date()} → "
      f"{train['date'].max().date()})")
print(f"Val   : {len(val):,} rows "
      f"({val['date'].min().date()} → "
      f"{val['date'].max().date()})")
print(f"Test  : {len(test):,} rows "
      f"({test['date'].min().date()} → "
      f"{test['date'].max().date()})")
print("\n✅ No data leakage enforced")

In [ ]:
# ── Features and target ───────────────────────────────────────────
TARGET = 'sales'

# Drop non-numeric columns that XGBoost cannot handle
drop_cols = ['date', 'store_id', 'product_id',
             'dataset', 'segment', 'abc_class',
             'xyz_class', 'Sales Quantity',
             'Price_x', 'Expiry Date']

# Build clean feature list — only numeric columns
numeric_cols = df.select_dtypes(
    include=['int64', 'float64', 'int32',
             'float32', 'bool', 'uint8']
).columns.tolist()

# Remove target and any leakage columns
ML_FEATURES = [c for c in numeric_cols
               if c not in [TARGET, 'sales_original',
                             'is_outlier_iqr',
                             'is_outlier_iso']]

print(f"ML Features: {len(ML_FEATURES)} columns")
print("Sample features:")
for f in ML_FEATURES[:10]:
    print(f"  {f}  — dtype: {df[f].dtype}")

X_train = train[ML_FEATURES].fillna(0)
y_train = train[TARGET]
X_val   = val[ML_FEATURES].fillna(0)
y_val   = val[TARGET]
X_test  = test[ML_FEATURES].fillna(0)
y_test  = test[TARGET]

print(f"\nX_train: {X_train.shape}")
print(f"X_val  : {X_val.shape}")
print(f"X_test : {X_test.shape}")
print("\n✅ All features confirmed numeric")
print("✅ No string columns — XGBoost ready")

In [ ]:
# ── Evaluation metrics function ───────────────────────────────────
def evaluate_model(y_true, y_pred, model_name):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mask = y_true != 0
    mape = np.mean(np.abs(
        (y_true[mask] - y_pred[mask]) / y_true[mask]
    )) * 100

    smape = np.mean(
        2 * np.abs(y_true - y_pred) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    results = {
        'Model': model_name,
        'MAE'  : round(mae,   4),
        'RMSE' : round(rmse,  4),
        'MAPE' : round(mape,  2),
        'sMAPE': round(smape, 2)
    }

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  MAPE  : {mape:.2f}%")
    print(f"  sMAPE : {smape:.2f}%")
    return results

all_results = []
print("✅ Metrics function ready")

In [ ]:
# ════════════════════════════════════════
# MODEL 1: NAIVE BASELINE
# ════════════════════════════════════════
print("Training Naive Baseline...")
start = time.time()

naive_pred_test = X_test['sales_lag_1'].fillna(
    y_train.mean()
)

elapsed = time.time() - start

with mlflow.start_run(run_name="Naive_Baseline"):
    mlflow.log_metric("MAE",   mean_absolute_error(
        y_test, naive_pred_test))
    mlflow.log_metric("RMSE",  np.sqrt(mean_squared_error(
        y_test, naive_pred_test)))
    mlflow.log_param("model_type", "baseline")

print(f"⏱ Time: {elapsed:.1f} seconds")
results_naive = evaluate_model(
    y_test, naive_pred_test, "Naive Baseline"
)
results_naive['Time_sec'] = round(elapsed, 2)
all_results.append(results_naive)
test = test.copy()
test['pred_naive'] = naive_pred_test.values
print("✅ Naive Baseline complete")

In [ ]:
# ════════════════════════════════════════
# MODEL 2: HOLT-WINTERS ETS
# FIX — replaced SimpleExpSmoothing
# ════════════════════════════════════════
from statsmodels.tsa.holtwinters import ExponentialSmoothing

print("Training Holt-Winters ETS...")
start = time.time()

daily_train = train.groupby('date')['sales'].mean()\
                   .reset_index()
daily_test  = test.groupby('date')['sales'].mean()\
                  .reset_index()

try:
    ets_model = ExponentialSmoothing(
        daily_train['sales'].values,
        trend            = 'add',
        seasonal         = 'add',
        seasonal_periods = 7,
        initialization_method = 'estimated'
    )
    ets_fit = ets_model.fit(optimized=True)
    print("✅ Holt-Winters (trend+seasonal) converged")
except Exception as e:
    print(f"Holt-Winters failed: {e}")
    print("Falling back to SimpleExpSmoothing...")
    from statsmodels.tsa.holtwinters import SimpleExpSmoothing
    ets_model = SimpleExpSmoothing(
        daily_train['sales'].values,
        initialization_method='estimated'
    )
    ets_fit = ets_model.fit(optimized=True)

ets_forecast = ets_fit.forecast(len(daily_test))
ets_pred_test = pd.Series(
    [ets_forecast[0]] * len(test),
    index=test.index
)

elapsed = time.time() - start

with mlflow.start_run(run_name="HoltWinters_ETS"):
    mlflow.log_metric("MAE", mean_absolute_error(
        y_test, ets_pred_test))
    mlflow.log_metric("RMSE", np.sqrt(mean_squared_error(
        y_test, ets_pred_test)))
    mlflow.log_param("model_type", "holt_winters")
    mlflow.log_param("trend", "add")
    mlflow.log_param("seasonal", "add")

print(f"⏱ Time: {elapsed:.1f} seconds")
results_ets = evaluate_model(
    y_test, ets_pred_test, "Holt-Winters ETS"
)
results_ets['Time_sec'] = round(elapsed, 2)
all_results.append(results_ets)
test['pred_ets'] = ets_pred_test.values
print("✅ Holt-Winters ETS complete")

In [ ]:
# ════════════════════════════════════════
# MODEL 3: ARIMA + SARIMA
# ════════════════════════════════════════
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("Training ARIMA + SARIMA...")
start = time.time()

daily_vals = daily_train['sales'].values

# ARIMA
try:
    arima_fit  = ARIMA(daily_vals,
                       order=(1,1,1)).fit()
    arima_fc   = arima_fit.forecast(len(daily_test))
    arima_pred = pd.Series(
        [arima_fc[0]] * len(test), index=test.index)
    print("✅ ARIMA(1,1,1) done")
except Exception as e:
    print(f"ARIMA failed: {e}")
    arima_pred = pd.Series(
        [y_train.mean()] * len(test), index=test.index)

# SARIMA
try:
    sarima_fit = SARIMAX(
        daily_vals,
        order=(1,1,1),
        seasonal_order=(1,1,0,7)
    ).fit(disp=False)
    sarima_fc   = sarima_fit.forecast(len(daily_test))
    sarima_pred = pd.Series(
        [sarima_fc[0]] * len(test), index=test.index)
    print("✅ SARIMA(1,1,1)(1,1,0,7) done")
except Exception as e:
    print(f"SARIMA failed: {e}")
    sarima_pred = arima_pred.copy()

elapsed = time.time() - start

with mlflow.start_run(run_name="ARIMA"):
    mlflow.log_metric("MAE", mean_absolute_error(
        y_test, arima_pred))
    mlflow.log_param("order", "(1,1,1)")

with mlflow.start_run(run_name="SARIMA"):
    mlflow.log_metric("MAE", mean_absolute_error(
        y_test, sarima_pred))
    mlflow.log_param("order", "(1,1,1)(1,1,0,7)")

print(f"⏱ Time: {elapsed:.1f} seconds")
results_arima = evaluate_model(
    y_test, arima_pred, "ARIMA")
results_arima['Time_sec'] = round(elapsed, 2)
all_results.append(results_arima)

results_sarima = evaluate_model(
    y_test, sarima_pred, "SARIMA")
results_sarima['Time_sec'] = round(elapsed, 2)
all_results.append(results_sarima)

test['pred_arima']  = arima_pred.values
test['pred_sarima'] = sarima_pred.values
print("✅ ARIMA + SARIMA complete")

In [ ]:
# ════════════════════════════════════════
# MODEL 4: XGBOOST + OPTUNA TUNING
# ════════════════════════════════════════
import xgboost as xgb

print("Tuning XGBoost with Optuna (30 trials)...")
start = time.time()

def objective_xgb(trial):
    params = {
        'n_estimators'    : trial.suggest_int(
                             'n_estimators', 100, 800),
        'max_depth'       : trial.suggest_int(
                             'max_depth', 3, 8),
        'learning_rate'   : trial.suggest_float(
                             'learning_rate', 0.01, 0.3),
        'subsample'       : trial.suggest_float(
                             'subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float(
                             'colsample_bytree', 0.6, 1.0),
        'random_state'    : 42,
        'n_jobs'          : -1,
        'verbosity'       : 0
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_train, y_train, verbose=False)
    return mean_absolute_error(y_val, m.predict(X_val))

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30)

best_xgb = study_xgb.best_params
print(f"\n✅ Best XGBoost params: {best_xgb}")
print(f"   Best Val MAE: {study_xgb.best_value:.4f}")

# Train final model with best params
xgb_model = xgb.XGBRegressor(
    **best_xgb, random_state=42,
    n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train, y_train, verbose=False)
xgb_pred_test = xgb_model.predict(X_test)

elapsed = time.time() - start

with mlflow.start_run(run_name="XGBoost_Optuna"):
    mlflow.log_params(best_xgb)
    mlflow.log_metric("MAE", mean_absolute_error(
        y_test, xgb_pred_test))
    mlflow.log_metric("RMSE", np.sqrt(mean_squared_error(
        y_test, xgb_pred_test)))
    mlflow.sklearn.log_model(xgb_model, "xgboost_model")

print(f"⏱ Time: {elapsed/60:.1f} minutes")
results_xgb = evaluate_model(
    y_test, xgb_pred_test, "XGBoost+Optuna")
results_xgb['Time_sec'] = round(elapsed, 2)
all_results.append(results_xgb)
test['pred_xgb'] = xgb_pred_test
pickle.dump(xgb_model,
    open(str(MODELS_PATH / 'xgboost_model.pkl'), 'wb'))
print("✅ XGBoost complete")

In [ ]:
# ════════════════════════════════════════
# MODEL 5: LIGHTGBM + OPTUNA TUNING
# ════════════════════════════════════════
import lightgbm as lgb

print("Tuning LightGBM with Optuna (30 trials)...")
start = time.time()

def objective_lgb(trial):
    params = {
        'n_estimators'    : trial.suggest_int(
                             'n_estimators', 100, 800),
        'max_depth'       : trial.suggest_int(
                             'max_depth', 3, 8),
        'learning_rate'   : trial.suggest_float(
                             'learning_rate', 0.01, 0.3),
        'subsample'       : trial.suggest_float(
                             'subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float(
                             'colsample_bytree', 0.6, 1.0),
        'random_state'    : 42,
        'n_jobs'          : -1,
        'verbose'         : -1
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(30, verbose=False),
                     lgb.log_evaluation(period=-1)])
    return mean_absolute_error(y_val, m.predict(X_val))

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30)

best_lgb = study_lgb.best_params
print(f"\n✅ Best LightGBM params: {best_lgb}")
print(f"   Best Val MAE: {study_lgb.best_value:.4f}")

lgb_model = lgb.LGBMRegressor(
    **best_lgb, random_state=42,
    n_jobs=-1, verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)
lgb_pred_test = lgb_model.predict(X_test)

elapsed = time.time() - start

with mlflow.start_run(run_name="LightGBM_Optuna"):
    mlflow.log_params(best_lgb)
    mlflow.log_metric("MAE", mean_absolute_error(
        y_test, lgb_pred_test))
    mlflow.log_metric("RMSE", np.sqrt(mean_squared_error(
        y_test, lgb_pred_test)))
    mlflow.sklearn.log_model(lgb_model, "lightgbm_model")

print(f"⏱ Time: {elapsed/60:.1f} minutes")
results_lgb = evaluate_model(
    y_test, lgb_pred_test, "LightGBM+Optuna")
results_lgb['Time_sec'] = round(elapsed, 2)
all_results.append(results_lgb)
test['pred_lgb'] = lgb_pred_test
pickle.dump(lgb_model,
    open(str(MODELS_PATH / 'lightgbm_model.pkl'), 'wb'))
print("✅ LightGBM complete")

In [ ]:
# ════════════════════════════════════════
# MODEL 6: LSTM — SCALER BUG FIXED
# ════════════════════════════════════════
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

print("Training LSTM...")
start = time.time()

# ── Scale — FIT ONLY ON TRAIN ─────────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit ONLY on training data
X_train_sc = scaler_X.fit_transform(X_train)
y_train_sc = scaler_y.fit_transform(
    y_train.values.reshape(-1,1)).flatten()

# TRANSFORM only — never fit on val/test
X_val_sc   = scaler_X.transform(X_val)        # ✅ fixed
y_val_sc   = scaler_y.transform(
    y_val.values.reshape(-1,1)).flatten()      # ✅ fixed

X_test_sc  = scaler_X.transform(X_test)       # ✅ fixed
y_test_sc  = scaler_y.transform(
    y_test.values.reshape(-1,1)).flatten()     # ✅ fixed

# Save scaler for Phase 4
import joblib
joblib.dump(scaler_X,
    str(MODELS_PATH / 'scaler_X.pkl'))
joblib.dump(scaler_y,
    str(MODELS_PATH / 'scaler_y.pkl'))
print("✅ Scalers saved — fit on train only")

# ── Tensors ───────────────────────────────────────────────────────
def to_ds(X, y):
    return TensorDataset(
        torch.FloatTensor(X).unsqueeze(1),
        torch.FloatTensor(y)
    )

train_dl = DataLoader(to_ds(X_train_sc, y_train_sc),
                      batch_size=64, shuffle=False)
val_dl   = DataLoader(to_ds(X_val_sc,   y_val_sc),
                      batch_size=64, shuffle=False)

# ── LSTM Model ────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size,
                 hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print(f"Device: {device}")

model     = LSTMModel(len(ML_FEATURES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# ── Training ──────────────────────────────────────────────────────
EPOCHS        = 30
best_val_loss = float('inf')
best_state    = None

for epoch in range(EPOCHS):
    model.train()
    for Xb, yb in train_dl:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(model(Xb), yb).backward()
        optimizer.step()

    model.eval()
    val_losses = []
    with torch.no_grad():
        for Xb, yb in val_dl:
            Xb, yb = Xb.to(device), yb.to(device)
            val_losses.append(
                criterion(model(Xb), yb).item())
    vl = np.mean(val_losses)

    if vl < best_val_loss:
        best_val_loss = vl
        best_state    = {k: v.clone()
                         for k, v in
                         model.state_dict().items()}

    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS} "
              f"val_loss: {vl:.6f}")

# ── Predict ───────────────────────────────────────────────────────
model.load_state_dict(best_state)
model.eval()
Xt = torch.FloatTensor(X_test_sc).unsqueeze(1).to(device)

with torch.no_grad():
    lstm_sc = model(Xt).cpu().numpy()

lstm_pred = scaler_y.inverse_transform(
    lstm_sc.reshape(-1,1)).flatten()

elapsed = time.time() - start

with mlflow.start_run(run_name="LSTM"):
    mlflow.log_param("epochs",     EPOCHS)
    mlflow.log_param("hidden",     64)
    mlflow.log_param("layers",     2)
    mlflow.log_param("dropout",    0.2)
    mlflow.log_metric("MAE",  mean_absolute_error(
        y_test, lstm_pred))
    mlflow.log_metric("RMSE", np.sqrt(mean_squared_error(
        y_test, lstm_pred)))
    mlflow.log_param("scaler_leak", "fixed")

print(f"\n⏱ Time: {elapsed/60:.1f} minutes")
results_lstm = evaluate_model(
    y_test, lstm_pred, "LSTM")
results_lstm['Time_sec'] = round(elapsed, 2)
all_results.append(results_lstm)
test['pred_lstm'] = lstm_pred
torch.save(best_state,
    str(MODELS_PATH / 'lstm_model.pt'))
print("✅ LSTM complete — scaler bug fixed")

In [ ]:
# ── Fix: Reinstall Darts with PyTorch support ─────────────────────
import subprocess, sys

print("Reinstalling Darts with PyTorch support...")

subprocess.check_call([sys.executable, '-m', 'pip',
    'uninstall', 'darts', '-y'])

subprocess.check_call([sys.executable, '-m', 'pip',
    'install', 'darts[torch]', '--quiet'])

print("✅ Darts reinstalled with PyTorch support")
print("Now restart kernel and run Cell 12")

In [ ]:
# MODEL 7: TFT — Pure PyTorch (no Darts)
import time                          # ← ADD THIS
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib

print("Training TFT (PyTorch implementation)...")
start = time.time()
# ... rest of your code unchanged

In [ ]:
# ════════════════════════════════════════
# MODEL 7: TFT — Pure PyTorch (no Darts)
# ════════════════════════════════════════
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib

print("Training TFT (PyTorch implementation)...")
start = time.time()

try:
    # ── Simple Transformer for time series ───────────────────
    class SimpleTFT(nn.Module):
        def __init__(self, input_size, d_model=32,
                     nhead=2, num_layers=2, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(input_size, d_model)
            encoder_layer   = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead,
                dropout=dropout, batch_first=True
            )
            self.transformer = nn.TransformerEncoder(
                encoder_layer, num_layers=num_layers
            )
            self.output = nn.Linear(d_model, 1)

        def forward(self, x):
            x = self.input_proj(x)
            x = self.transformer(x)
            return self.output(x[:, -1, :]).squeeze()

    # ── Scale data ────────────────────────────────────────────
    scaler_tft = MinMaxScaler()
    X_tr_sc = scaler_tft.fit_transform(X_train)
    X_v_sc  = scaler_tft.transform(X_val)
    X_te_sc = scaler_tft.transform(X_test)

    scaler_ty = MinMaxScaler()
    y_tr_sc = scaler_ty.fit_transform(
        y_train.values.reshape(-1,1)).flatten()
    y_te_sc = scaler_ty.transform(
        y_test.values.reshape(-1,1)).flatten()

    # ── Tensors ───────────────────────────────────────────────
    from torch.utils.data import DataLoader, TensorDataset

    def make_dl(X, y, bs=64):
        ds = TensorDataset(
            torch.FloatTensor(X).unsqueeze(1),
            torch.FloatTensor(y)
        )
        return DataLoader(ds, batch_size=bs,
                          shuffle=False)

    train_dl_t = make_dl(X_tr_sc, y_tr_sc)

    # ── Train ─────────────────────────────────────────────────
    device  = torch.device('cpu')
    tft_m   = SimpleTFT(len(ML_FEATURES)).to(device)
    opt     = torch.optim.Adam(tft_m.parameters(),
                                lr=0.001)
    crit    = nn.MSELoss()
    EPOCHS  = 30
    best_st = None
    best_vl = float('inf')

    for epoch in range(EPOCHS):
        tft_m.train()
        for Xb, yb in train_dl_t:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(tft_m(Xb), yb).backward()
            opt.step()

        # Validation loss
        tft_m.eval()
        with torch.no_grad():
            Xv = torch.FloatTensor(X_v_sc)\
                      .unsqueeze(1).to(device)
            yv = torch.FloatTensor(
                scaler_ty.transform(
                    y_val.values.reshape(-1,1)
                ).flatten()).to(device)
            vl = crit(tft_m(Xv), yv).item()

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.clone() for k, v in
                       tft_m.state_dict().items()}

        if (epoch+1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} "
                  f"val_loss: {vl:.6f}")

    # ── Predict ───────────────────────────────────────────────
    tft_m.load_state_dict(best_st)
    tft_m.eval()
    Xt = torch.FloatTensor(X_te_sc)\
              .unsqueeze(1).to(device)
    with torch.no_grad():
        tft_sc = tft_m(Xt).cpu().numpy()

    tft_pred = scaler_ty.inverse_transform(
        tft_sc.reshape(-1,1)).flatten()

    elapsed = time.time() - start

    with mlflow.start_run(run_name="TFT_Transformer"):
        mlflow.log_param("model_type",  "transformer")
        mlflow.log_param("d_model",     32)
        mlflow.log_param("nhead",       2)
        mlflow.log_param("num_layers",  2)
        mlflow.log_param("epochs",      EPOCHS)
        mlflow.log_metric("MAE", mean_absolute_error(
            y_test, tft_pred))
        mlflow.log_metric("RMSE", np.sqrt(
            mean_squared_error(y_test, tft_pred)))

    print(f"\n⏱ Time: {elapsed/60:.1f} minutes")
    results_tft = evaluate_model(
        y_test, tft_pred, "TFT (Transformer)")
    results_tft['Time_sec'] = round(elapsed, 2)
    all_results.append(results_tft)
    test['pred_tft'] = tft_pred
    torch.save(best_st,
        str(MODELS_PATH / 'tft_model.pt'))
    print("✅ TFT complete")

except Exception as e:
    print(f"⚠️ TFT failed: {e}")
    all_results.append({
        'Model': 'TFT', 'MAE': None,
        'RMSE': None, 'MAPE': None,
        'sMAPE': None, 'Time_sec': None
    })